# F1-M05: Dashboard Fase 1

**TFM: Predicción de Abandono Universitario**

| | |
|---|---|
| **Autora** | María José Morte |
| **Email** | mjmorteruiz@uoc.edu (UOC) \| morte@uji.es (UJI) |

---

## 🎯 ¿Qué hace este notebook?

Genera un **dashboard HTML** que resume visualmente toda la Fase 1
(transformación de datos). Incluye:

- KPIs principales (tablas, registros, variables, alumnos)
- Gráfico de barras SVG con registros por tabla
- Donut chart de columnas por origen (Excel vs Preinscripción)
- Pipeline visual de la Fase 1 (M01 → M04)
- Grid de transformaciones aplicadas
- Listado de tablas origen con emojis y estadísticas

Todo se calcula dinámicamente desde los datos reales.

## ⚠️ Requisitos

- Haber ejecutado M02 (parquets en `01_interim/`)
- Haber ejecutado M04 (df_alumno.parquet en `02_processed/`)

## 📦 ¿Qué genera?

| Archivo | Ubicación | Descripción |
|---|---|---|
| m05_dashboard.html | `docs/html/fase1/` | Dashboard interactivo de la Fase 1 |

## 📌 Siguiente

- `f1_m06_grafo.ipynb` (grafo de relaciones entre tablas)

In [1]:
# ============================================================================
# CELDA 1: CONFIGURACIÓN
# ============================================================================
#
# Importa rutas y constantes del proyecto.
#
# De src.constantes importamos las constantes de presentación:
#   - EMOJIS_TABLAS / COLORES_TABLAS: identidad visual de cada tabla
#   - PIPELINE_FASE1: etapas del pipeline (M01→M04)
#   - TRANSFORMACIONES_FASE1: lista de transformaciones aplicadas
#   - COLORES_KPIS: colores para los indicadores principales
# ============================================================================

import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
import pandas as pd

# --- Detectar entorno ---
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/drive/MyDrive/AU_UJI')
else:
    ROOT = Path.cwd()
    for _ in range(5):
        if (ROOT / 'src').exists():
            break
        ROOT = ROOT.parent

if not (ROOT / 'src').exists():
    raise FileNotFoundError(
        f'No se encontró la carpeta src/ en {ROOT}\n'
        f'Asegúrate de que este notebook está dentro del proyecto (AU_UJI/)'
    )

sys.path.insert(0, str(ROOT))

# --- Imports del proyecto ---
from src.config import RUTA_INTERIM, RUTA_PROCESSED, RUTA_HTML, info_entorno
from src.utils import formato_numero_es
from src.constantes import (
    TABLA_PREINSCRIPCION,
    EMOJIS_TABLAS,
    COLORES_TABLAS,
    PIPELINE_FASE1,
    TRANSFORMACIONES_FASE1,
    COLORES_KPIS
)
from src.html import generar_html_navegacion_completa, guardar_html
from src.html.render import render_base_html

# --- Carpeta de salida ---
RUTA_FASE1 = RUTA_HTML / 'fase1'
RUTA_FASE1.mkdir(parents=True, exist_ok=True)

# Atajo para formato español
fmt = formato_numero_es

info_entorno()

✓ ===========================================================================
✓ 📌 INFORMACIÓN DEL ENTORNO DEL PROYECTO
✓ ===========================================================================
✓ 🖥️  Entorno detectado: Local
✓ 📂 Ruta base:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
✓ 📁 RAW:           C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw
✓ 📁 INTERIM:       C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\01_interim
✓ 📁 PROCESSED:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\02_processed
✓ 📁 FEATURES:      C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\03_features
✓ 📁 AUTOML:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\automl
✓ 📁 NOTEBOOKS:     C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\notebooks
✓ 📄 Excel principal: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\data\00_raw\datos_proyecto_sin_preinscrip.xlsx
✓ ============================

In [2]:
# ============================================================================
# CELDA 2: CARGAR DATOS Y CALCULAR MÉTRICAS
# ============================================================================
#
# Lee dinámicamente todos los parquets de interim para obtener
# las métricas de cada tabla (filas, columnas).
#
# También carga df_alumno.parquet para los KPIs globales.
#
# Las métricas se separan por origen:
#   - tabs_excel: tablas del Excel principal (8 tablas)
#   - tabs_preins: tabla de preinscripción (1 tabla)
# Esto se usa para el donut chart de columnas por origen.
# ============================================================================

print('=' * 60)
print('CARGANDO DATOS')
print('=' * 60)

# --- Métricas de cada tabla individual ---
TABLAS = {}
for pq in sorted(RUTA_INTERIM.glob('*.parquet')):
    df = pd.read_parquet(pq)
    TABLAS[pq.stem] = {
        'filas': len(df),
        'n_cols': len(df.columns),
        'es_preins': pq.stem == TABLA_PREINSCRIPCION
    }
    print(f'  ✅ {pq.stem:15s}: {fmt(len(df)):>10s} × {len(df.columns)} cols')

# --- Dataset final ---
df_path = RUTA_PROCESSED / 'df_alumno.parquet'
if df_path.exists():
    df = pd.read_parquet(df_path)
    n_registros = len(df)
    n_columnas = len(df.columns)
    n_alumnos = df['per_id_ficticio'].nunique()
else:
    n_registros, n_columnas, n_alumnos = 0, 0, 0

# --- Métricas por origen ---
n_tablas = len(TABLAS)
tabs_excel = [t for t in TABLAS if not TABLAS[t]['es_preins']]
tabs_preins = [t for t in TABLAS if TABLAS[t]['es_preins']]
cols_excel = sum(TABLAS[t]['n_cols'] for t in tabs_excel)
cols_preins = sum(TABLAS[t]['n_cols'] for t in tabs_preins)

print(f'\n🎯 df_alumno: {fmt(n_registros)} × {n_columnas} columnas')
print(f'   Alumnos únicos: {fmt(n_alumnos)}')

CARGANDO DATOS
  ✅ becas          :     70.524 × 4 cols
  ✅ demograficos   :     30.873 × 5 cols
  ✅ domicilios     :    210.911 × 6 cols
  ✅ expedientes    :    109.568 × 15 cols
  ✅ notas          :    107.908 × 5 cols
  ✅ preinscripcion :    210.986 × 24 cols
  ✅ recibos        :    114.454 × 5 cols
  ✅ titulaciones   :         45 × 6 cols
  ✅ trabajo        :    195.524 × 4 cols

🎯 df_alumno: 109.568 × 37 columnas
   Alumnos únicos: 30.872


In [3]:
# ============================================================================
# CELDA 3: GENERAR HTML DASHBOARD
# ============================================================================
#
# Construye el dashboard m05_dashboard.html con:
#   1. KPIs principales (tablas, registros, variables, alumnos)
#   2. Gráfico de barras SVG: registros por tabla (dinámico)
#   3. Donut chart: columnas Excel vs Preinscripción (dinámico)
#   4. Pipeline visual: etapas de la Fase 1 (desde PIPELINE_FASE1)
#   5. Grid de transformaciones (desde TRANSFORMACIONES_FASE1)
#   6. Listado de tablas con emoji y color (desde constantes)
#
# Todo se genera dinámicamente — los valores vienen de las variables
# calculadas en la celda anterior, los emojis y colores de constantes.py.
#
# render_base_html() envuelve todo con la plantilla Jinja2 (base.html)
# que incluye cabecera con logos, navegación y footer.
# ============================================================================

nav_fases, nav_modulos = generar_html_navegacion_completa(fase_activa='fase1', modulo_activo='m05')
fmt = formato_numero_es

# ---- KPIs ----
kpis_html = f'''
<div class="dashboard-kpis">
    <div class="dash-kpi">
        <div class="dash-kpi-valor">{n_tablas}</div>
        <div class="dash-kpi-label">Tablas</div>
        <div class="dash-kpi-icon" style="background:{COLORES_KPIS['tablas']}">📊</div>
    </div>
    <div class="dash-kpi">
        <div class="dash-kpi-valor">{fmt(n_registros)}</div>
        <div class="dash-kpi-label">Registros</div>
        <div class="dash-kpi-icon" style="background:{COLORES_KPIS['registros']}">📋</div>
    </div>
    <div class="dash-kpi">
        <div class="dash-kpi-valor">{n_columnas}</div>
        <div class="dash-kpi-label">Variables</div>
        <div class="dash-kpi-icon" style="background:{COLORES_KPIS['variables']}">📐</div>
    </div>
    <div class="dash-kpi">
        <div class="dash-kpi-valor">{fmt(n_alumnos)}</div>
        <div class="dash-kpi-label">Alumnos</div>
        <div class="dash-kpi-icon" style="background:{COLORES_KPIS['alumnos']}">👤</div>
    </div>
</div>
'''

# ---- Barras SVG ----
barras_labels = sorted(TABLAS.keys())
barras_valores = [TABLAS[t]['filas'] for t in barras_labels]
barras_colores = [COLORES_TABLAS.get(t, '#3182ce') for t in barras_labels]
max_filas = max(barras_valores)

barras_svg = ''
for i, (label, valor, color) in enumerate(zip(barras_labels, barras_valores, barras_colores)):
    altura = int((valor / max_filas) * 150)
    y = 160 - altura
    barras_svg += f'<g><rect x="{30 + i*75}" y="{y}" width="55" height="{altura}" fill="{color}" rx="4"/><text x="{57 + i*75}" y="175" text-anchor="middle" font-size="9" fill="#4a5568">{label[:6]}</text><text x="{57 + i*75}" y="{y-5}" text-anchor="middle" font-size="8" fill="#718096">{valor//1000}k</text></g>'

# ---- Donut ----
total_cols = cols_excel + cols_preins
pct_excel = (cols_excel / total_cols * 100) if total_cols > 0 else 0

# ---- Pipeline ----
pipeline_html = ''
for i, paso in enumerate(PIPELINE_FASE1):
    if i > 0:
        pipeline_html += '<div class="dash-pipe-arrow">→</div>'
    pipeline_html += f'<div class="dash-pipe-step" style="--step-color:{paso["color"]}"><div class="dash-pipe-icon">{paso["emoji"]}</div><div class="dash-pipe-text">{paso["id"]}<br><small>{paso["nombre"]}</small></div></div>'

# ---- Transformaciones ----
trans_html = ''
for t in TRANSFORMACIONES_FASE1:
    trans_html += f'<div class="dash-trans"><span>{t["emoji"]}</span> {t["nombre"]}</div>'

# ---- Tablas ----
tablas_html = ''
for t in sorted(TABLAS.keys()):
    info = TABLAS[t]
    emoji = EMOJIS_TABLAS.get(t, '📊')
    color = COLORES_TABLAS.get(t, '#3182ce')
    tablas_html += f'<div class="dash-tabla-item" style="border-left-color:{color}"><span class="dash-tabla-emoji">{emoji}</span><div class="dash-tabla-info"><div class="dash-tabla-nombre">{t}</div><div class="dash-tabla-stats">{fmt(info["filas"])} × {info["n_cols"]}</div></div></div>'

# ---- Contenido ----
contenido = f'''
<div class="dashboard-grid">
    <div class="dash-card"><div class="dash-card-header"><h3>📊 Registros por Tabla</h3></div><div class="dash-card-body"><svg viewBox="0 0 700 190" style="width:100%;height:180px;">{barras_svg}</svg></div></div>
    <div class="dash-card"><div class="dash-card-header"><h3>📐 Columnas por Origen</h3></div><div class="dash-card-body dash-donut-container"><div class="dash-donut" style="--pct-1:{pct_excel}; --color-1:#3182ce; --color-2:#805ad5;"><div class="dash-donut-center"><span class="dash-donut-valor">{total_cols}</span><span class="dash-donut-label">cols</span></div></div><div class="dash-donut-legend"><div class="dash-legend-item"><span style="background:#3182ce"></span>Excel ({cols_excel})</div><div class="dash-legend-item"><span style="background:#805ad5"></span>Preins ({cols_preins})</div></div></div></div>
    <div class="dash-card"><div class="dash-card-header"><h3>🔄 Pipeline</h3></div><div class="dash-card-body"><div class="dash-pipeline">{pipeline_html}</div></div></div>
    <div class="dash-card"><div class="dash-card-header"><h3>⚙️ Transformaciones</h3></div><div class="dash-card-body"><div class="dash-trans-grid">{trans_html}</div></div></div>
    <div class="dash-card dash-card-wide"><div class="dash-card-header"><h3>📁 Tablas Origen</h3></div><div class="dash-card-body"><div class="dash-tablas-grid">{tablas_html}</div></div></div>
</div>
'''

html = render_base_html(
    titulo='M05: Dashboard',
    icono='📊',
    subtitulo='Fase 1: Resumen de Transformación',
    nav_fases=nav_fases,
    nav_modulos=nav_modulos,
    contenido=kpis_html + contenido,
    notebook_nombre='f1_m05_dashboard.ipynb',
    notebook_carpeta='fase1_transformacion'
)

guardar_html(html, RUTA_FASE1 / 'm05_dashboard.html')
print('\n✅ m05_dashboard.html')

✅ HTML guardado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase1\m05_dashboard.html

✅ m05_dashboard.html
